## SRP044761 / GSE59707

**paper:** [no paper, seems likely that data is described in this thesis](https://kuleuven.limo.libis.be/discovery/fulldisplay?docid=lirias1757736&context=SearchWebhook&vid=32KUL_KUL:Lirias&lang=en&search_scope=lirias_profile&adaptor=SearchWebhook&tab=LIRIAS&query=any,contains,LIRIAS1757736&offset=0) 

**date, curator:** 2026-06-23, Sara Carsanaro

**notes**
* unfortunately no paper however most of the sample details can be found in SRA/GEO
* did manual annotation because FBbt and FBdv stages needed for 

### annotation summary

In [20]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,brain,UBERON:6001920,insect embryonic/larval brain,perfect match
7,brain,FBbt:00001920,larval brain,perfect match
9,wing imaginal disc,UBERON:6001778,insect wing disc,perfect match
16,wing imaginal disc,FBbt:00001778,wing disc,perfect match
18,eye-antennal imaginal disc,UBERON:6001766,insect eye-antennal disc,perfect match
25,eye-antennal imaginal disc,FBbt:00001766,eye-antennal disc,perfect match


In [21]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,3rd instar larvae,UBERON:8000002,third instar larva stage,perfect match
7,3rd instar larvae,FBdv:00005339,third instar larval stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP044761"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX660302,SRP044761,Illumina HiSeq 2000,SRS665300,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444035,brain,3rd instar larvae,perfect match,not documented,perfect match,,,,7244,,,,,,RNA_Dvir_brain,"SAMN02934586,GSM1444035",,,,,,,,,,,23/06/2026,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
1,SRX660301,SRP044761,Illumina HiSeq 2000,SRS665299,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444034,brain,3rd instar larvae,perfect match,not documented,perfect match,,,,7230,,,,,,RNA_Dmoj_brain,"SAMN02934585,GSM1444034",,,,,,,,,,,23/06/2026,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
2,SRX660300,SRP044761,Illumina HiSeq 2000,SRS665298,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444033,brain,3rd instar larvae,perfect match,not documented,perfect match,,,,7260,,,,,,RNA_Dwil_brain,"SAMN02934584,GSM1444033",,,,,,,,,,,23/06/2026,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
3,SRX660299,SRP044761,Illumina HiSeq 2000,SRS665301,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444032,brain,3rd instar larvae,perfect match,not documented,perfect match,,,,7237,,,,,,RNA_Dpse_brain,"SAMN02934583,GSM1444032",,,,,,,,,,,23/06/2026,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
4,SRX660298,SRP044761,Illumina HiSeq 2000,SRS665297,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444031,brain,3rd instar larvae,perfect match,not documented,perfect match,,,,7217,,,,,,RNA_Dana_brain,"SAMN02934582,GSM1444031",,,,,,,,,,,23/06/2026,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
5,SRX660297,SRP044761,Illumina HiSeq 2000,SRS665296,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444030,brain,3rd instar larvae,perfect match,not documented,perfect match,,,,7245,,,,,,RNA_Dyak_brain,"SAMN02934581,GSM1444030",,,,,,,,,,,23/06/2026,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
6,SRX660296,SRP044761,Illumina HiSeq 2000,SRS665294,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444029,brain,3rd instar larvae,perfect match,not documented,perfect match,,,,7240,,,,,,RNA_Dsim_brain,"SAMN02934580,GSM1444029",,,,,,,,,,,23/06/2026,RNA was extracted with t

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['brain' 'eye-antennal imaginal disc' 'wing imaginal disc']


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [6]:
unique_sorted(library, "infoStage")

['3rd instar larvae']


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [7]:
library.loc[:,'sex'] = 'NA'
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX660302,SRP044761,Illumina HiSeq 2000,SRS665300,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444035,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7244,,,,,,RNA_Dvir_brain,"SAMN02934586,GSM1444035",,,,,,,,,,,23/06/2026,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
1,SRX660301,SRP044761,Illumina HiSeq 2000,SRS665299,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444034,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7230,,,,,,RNA_Dmoj_brain,"SAMN02934585,GSM1444034",,,,,,,,,,,23/06/2026,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
2,SRX660300,SRP044761,Illumina HiSeq 2000,SRS665298,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444033,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7260,,,,,,RNA_Dwil_brain,"SAMN02934584,GSM1444033",,,,,,,,,,,23/06/2026,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
3,SRX660299,SRP044761,Illumina HiSeq 2000,SRS665301,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444032,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7237,,,,,,RNA_Dpse_brain,"SAMN02934583,GSM1444032",,,,,,,,,,,23/06/2026,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
4,SRX660298,SRP044761,Illumina HiSeq 2000,SRS665297,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444031,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7217,,,,,,RNA_Dana_brain,"SAMN02934582,GSM1444031",,,,,,,,,,,23/06/2026,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
5,SRX660297,SRP044761,Illumina HiSeq 2000,SRS665296,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444030,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7245,,,,,,RNA_Dyak_brain,"SAMN02934581,GSM1444030",,,,,,,,,,,23/06/2026,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
6,SRX660296,SRP044761,Illumina HiSeq 2000,SRS665294,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444029,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7240,,,,,,RNA_Dsim_brain,"SAMN02934580,GSM1444029",,,,,,,,,,,23/06/2026,RNA was ex

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [8]:
# making these variables because we use them again in the experiment file
#my_protocol = ''
# full_length or 3'
#my_protocolType = ''

#library.loc[:,'protocol'] = my_protocol
#library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX660302,SRP044761,Illumina HiSeq 2000,SRS665300,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444035,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7244,,,polyA,,,RNA_Dvir_brain,"SAMN02934586,GSM1444035",,,,,,,,,,,23/06/2026,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
1,SRX660301,SRP044761,Illumina HiSeq 2000,SRS665299,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444034,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7230,,,polyA,,,RNA_Dmoj_brain,"SAMN02934585,GSM1444034",,,,,,,,,,,23/06/2026,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
2,SRX660300,SRP044761,Illumina HiSeq 2000,SRS665298,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444033,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7260,,,polyA,,,RNA_Dwil_brain,"SAMN02934584,GSM1444033",,,,,,,,,,,23/06/2026,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
3,SRX660299,SRP044761,Illumina HiSeq 2000,SRS665301,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444032,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7237,,,polyA,,,RNA_Dpse_brain,"SAMN02934583,GSM1444032",,,,,,,,,,,23/06/2026,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
4,SRX660298,SRP044761,Illumina HiSeq 2000,SRS665297,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444031,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7217,,,polyA,,,RNA_Dana_brain,"SAMN02934582,GSM1444031",,,,,,,,,,,23/06/2026,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
5,SRX660297,SRP044761,Illumina HiSeq 2000,SRS665296,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444030,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7245,,,polyA,,,RNA_Dyak_brain,"SAMN02934581,GSM1444030",,,,,,,,,,,23/06/2026,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
6,SRX660296,SRP044761,Illumina HiSeq 2000,SRS665294,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444029,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7240,,,polyA,,,RNA_Dsim_brain,"SAMN02934580,GSM14440

#### globin, replicates

In [9]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [10]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-06-25'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX660302,SRP044761,Illumina HiSeq 2000,SRS665300,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444035,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7244,,,polyA,,,RNA_Dvir_brain,"SAMN02934586,GSM1444035",,,,,,,,,,SAC,2026-06-25,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
1,SRX660301,SRP044761,Illumina HiSeq 2000,SRS665299,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444034,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7230,,,polyA,,,RNA_Dmoj_brain,"SAMN02934585,GSM1444034",,,,,,,,,,SAC,2026-06-25,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
2,SRX660300,SRP044761,Illumina HiSeq 2000,SRS665298,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444033,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7260,,,polyA,,,RNA_Dwil_brain,"SAMN02934584,GSM1444033",,,,,,,,,,SAC,2026-06-25,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
3,SRX660299,SRP044761,Illumina HiSeq 2000,SRS665301,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444032,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7237,,,polyA,,,RNA_Dpse_brain,"SAMN02934583,GSM1444032",,,,,,,,,,SAC,2026-06-25,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
4,SRX660298,SRP044761,Illumina HiSeq 2000,SRS665297,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444031,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7217,,,polyA,,,RNA_Dana_brain,"SAMN02934582,GSM1444031",,,,,,,,,,SAC,2026-06-25,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
5,SRX660297,SRP044761,Illumina HiSeq 2000,SRS665296,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444030,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7245,,,polyA,,,RNA_Dyak_brain,"SAMN02934581,GSM1444030",,,,,,,,,,SAC,2026-06-25,RNA was extracted with the RNAqueous micro kit (Ambion) RNA libraries were generated for sequencing using standard Illumina TruSeq protocols,,,,brain,,,,TRANSCRIPTOMIC,cDNA
6,SRX660296,SRP044761,Illumina HiSeq 2000,SRS665294,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444029,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7240,,,polyA,,,RNA_Dsim_brain,"SAM

#### comments

In [ ]:
#library.loc[:,'comment'] = ''

#### save complete file with correct columns

In [11]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX660302,SRP044761,Illumina HiSeq 2000,SRS665300,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444035,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7244,,,polyA,,,RNA_Dvir_brain,"SAMN02934586,GSM1444035",,,,,,,,,,SAC,2026-06-25
1,SRX660301,SRP044761,Illumina HiSeq 2000,SRS665299,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444034,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7230,,,polyA,,,RNA_Dmoj_brain,"SAMN02934585,GSM1444034",,,,,,,,,,SAC,2026-06-25
2,SRX660300,SRP044761,Illumina HiSeq 2000,SRS665298,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444033,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7260,,,polyA,,,RNA_Dwil_brain,"SAMN02934584,GSM1444033",,,,,,,,,,SAC,2026-06-25
3,SRX660299,SRP044761,Illumina HiSeq 2000,SRS665301,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444032,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7237,,,polyA,,,RNA_Dpse_brain,"SAMN02934583,GSM1444032",,,,,,,,,,SAC,2026-06-25
4,SRX660298,SRP044761,Illumina HiSeq 2000,SRS665297,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444031,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7217,,,polyA,,,RNA_Dana_brain,"SAMN02934582,GSM1444031",,,,,,,,,,SAC,2026-06-25
5,SRX660297,SRP044761,Illumina HiSeq 2000,SRS665296,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444030,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7245,,,polyA,,,RNA_Dyak_brain,"SAMN02934581,GSM1444030",,,,,,,,,,SAC,2026-06-25
6,SRX660296,SRP044761,Illumina HiSeq 2000,SRS665294,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444029,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7240,,,polyA,,,RNA_Dsim_brain,"SAMN02934580,GSM1444029",,,,,,,,,,SAC,2026-06-25
7,SRX660295,SRP044761,Illumina HiSeq 2000,SRS665295,FBbt:00001920,larval brain,FBdv:00005339,third instar larval stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444028,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,Canton-S,7227,,,polyA,,,RNA_Dmel_CS_brain: Larval_brain_WT_cs,"SAMN02934579,GSM1444028",,,,,,,,,,SAC,2026-06-25
8,SRX660294,SRP044761,Illumina HiSeq 2000,SRS665293,FBbt:00001920,larval brain,FBdv:00005339,third instar larval stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444027,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,DGRP-208,7227,,,polyA,,,RNA_Dmel_RAL-208_brain: Larval_brain_WT_RAL-208,"SAMN02934578,GSM1444027",,,,,,,,,,SAC,2026-06-25
9,SRX660293,SRP044761,Illumina HiSeq 2000,SRS665292,UBERON:6001778,insect wing disc,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1444026,wing imaginal disc,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7244,,,polyA,,,RNA_Dvir_wing,"SAMN02934577,GSM1444026",,,,,,,,,,SAC,2026-06-25


### experiment annotations

In [12]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP044761,Evolutionary changes in cis regulation are associated with altered chromatin activity and gene expression levels in the Drosophila eye development (RNA-Seq),"Expression profiling by high-throughput sequencing (RNA-seq) across eight WT Drosophila species - D. melanogaster, D. simulans, D. yakuba, D. ananassae, D. pseudoobscura, D. willistoni, D. mojavensis and D. virilis - and three tissues, namely, brain, eye-antennal and imaginal discs, at the stage of third instar larvae. Overall design: By the use of Ornstein-Uhlenbeck methods, we assess the evolutionary forces acting on regulatory elements (cis-level), chromatin activity and gene expression across Drosophila eye-antennal imaginal discs at the stage of third instar larvae.",SRA,,,,,,GSE59707,PRJNA256000,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [13]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

27

In [14]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
#experiment.loc[:,'protocol'] = my_protocol
#experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP044761,Evolutionary changes in cis regulation are associated with altered chromatin activity and gene expression levels in the Drosophila eye development (RNA-Seq),"Expression profiling by high-throughput sequencing (RNA-seq) across eight WT Drosophila species - D. melanogaster, D. simulans, D. yakuba, D. ananassae, D. pseudoobscura, D. willistoni, D. mojavensis and D. virilis - and three tissues, namely, brain, eye-antennal and imaginal discs, at the stage of third instar larvae. Overall design: By the use of Ornstein-Uhlenbeck methods, we assess the evolutionary forces acting on regulatory elements (cis-level), chromatin activity and gene expression across Drosophila eye-antennal imaginal discs at the stage of third instar larvae.",SRA,total,Bgee 1K,27,,,GSE59707,PRJNA256000,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [15]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
#experiment.loc[:,'PMID'] = ''
experiment.loc[:,'reference_url'] = 'https://kuleuven.limo.libis.be/discovery/fulldisplay?docid=lirias1757736&context=SearchWebhook&vid=32KUL_KUL:Lirias&lang=en&search_scope=lirias_profile&adaptor=SearchWebhook&tab=LIRIAS&query=any,contains,LIRIAS1757736&offset=0'
experiment.loc[:,'DOI'] = ''
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP044761,Evolutionary changes in cis regulation are associated with altered chromatin activity and gene expression levels in the Drosophila eye development (RNA-Seq),"Expression profiling by high-throughput sequencing (RNA-seq) across eight WT Drosophila species - D. melanogaster, D. simulans, D. yakuba, D. ananassae, D. pseudoobscura, D. willistoni, D. mojavensis and D. virilis - and three tissues, namely, brain, eye-antennal and imaginal discs, at the stage of third instar larvae. Overall design: By the use of Ornstein-Uhlenbeck methods, we assess the evolutionary forces acting on regulatory elements (cis-level), chromatin activity and gene expression across Drosophila eye-antennal imaginal discs at the stage of third instar larvae.",SRA,total,Bgee 1K,27,,,GSE59707,PRJNA256000,,"https://kuleuven.limo.libis.be/discovery/fulldisplay?docid=lirias1757736&context=SearchWebhook&vid=32KUL_KUL:Lirias&lang=en&search_scope=lirias_profile&adaptor=SearchWebhook&tab=LIRIAS&query=any,contains,LIRIAS1757736&offset=0",,,


#### comments

In [16]:
experiment.loc[:,'comment'] = 'no paper, seems likely this is part of a thesis linked here'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP044761,Evolutionary changes in cis regulation are associated with altered chromatin activity and gene expression levels in the Drosophila eye development (RNA-Seq),"Expression profiling by high-throughput sequencing (RNA-seq) across eight WT Drosophila species - D. melanogaster, D. simulans, D. yakuba, D. ananassae, D. pseudoobscura, D. willistoni, D. mojavensis and D. virilis - and three tissues, namely, brain, eye-antennal and imaginal discs, at the stage of third instar larvae. Overall design: By the use of Ornstein-Uhlenbeck methods, we assess the evolutionary forces acting on regulatory elements (cis-level), chromatin activity and gene expression across Drosophila eye-antennal imaginal discs at the stage of third instar larvae.",SRA,total,Bgee 1K,27,,,GSE59707,PRJNA256000,,"https://kuleuven.limo.libis.be/discovery/fulldisplay?docid=lirias1757736&context=SearchWebhook&vid=32KUL_KUL:Lirias&lang=en&search_scope=lirias_profile&adaptor=SearchWebhook&tab=LIRIAS&query=any,contains,LIRIAS1757736&offset=0",,,"no paper, seems likely this is part of a thesis linked here"


#### save complete file

In [17]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [18]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [19]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [22]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [23]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
66101,SRX28647796,SRP582736,Illumina HiSeq 1500,SRS24918247,UBERON:0002101,limb,UBERON:0000104,life cycle,,Limb,Stage52,other,not documented,other,NA,NA,,8296,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,Model organism or animal sample from Ambystoma...,SAMN48288855,,,,,,,"PMID:41982182, For the three birds, fertilized...",,,ANN,2026-06-23
66102,SRX28647795,SRP582736,Illumina HiSeq 1500,SRS24918246,UBERON:0002101,limb,UBERON:0000104,life cycle,,Limb,Stage52,other,not documented,other,NA,NA,,8296,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,Model organism or animal sample from Ambystoma...,SAMN48288854,,,,,,,"PMID:41982182, For the three birds, fertilized...",,,ANN,2026-06-23
66103,SRX660302,SRP044761,Illumina HiSeq 2000,SRS665300,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7244,,,polyA,,,RNA_Dvir_brain,"SAMN02934586,GSM1444035",,,,,,,,,,SAC,2026-06-25
66104,SRX660301,SRP044761,Illumina HiSeq 2000,SRS665299,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7230,,,polyA,,,RNA_Dmoj_brain,"SAMN02934585,GSM1444034",,,,,,,,,,SAC,2026-06-25
66105,SRX660300,SRP044761,Illumina HiSeq 2000,SRS665298,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7260,,,polyA,,,RNA_Dwil_brain,"SAMN02934584,GSM1444033",,,,,,,,,,SAC,2026-06-25
66106,SRX660299,SRP044761,Illumina HiSeq 2000,SRS665301,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7237,,,polyA,,,RNA_Dpse_brain,"SAMN02934583,GSM1444032",,,,,,,,,,SAC,2026-06-25
66107,SRX660298,SRP044761,Illumina HiSeq 2000,SRS665297,UBERON:6001920,insect embryonic/larval brain,UBERON:8000002,third instar larva stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,brain,3rd instar larvae,perfect match,not documented,perfect match,NA,,,7217,,,polyA,,,RNA_Dana_brain,"SAMN02934582,GSM1444031",,,,,,,,,,SAC,2026-06-25


In [24]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1267,SRP026279,Quantitative genome-wide enhancer activity map...,Phenotypic differences between closely related...,SRA,partial,Bgee 1K,2,,,GSE48251,PRJNA209379,24908250,https://pmc.ncbi.nlm.nih.gov/articles/PMC4250274/,10.1038/ng.3009,,removed WGS and STARR-seq libraries
1268,SRP582736,Evolution of limb and digit identity genes sin...,The datasets are used to create the analysis o...,SRA,total,Bgee 1K,249,"NEBNext Ultra RNA Library Prep Kit,DNBSEQ-T7",full_length,,PRJNA1258050,41982182,https://pmc.ncbi.nlm.nih.gov/articles/PMC13135...,10.1093/molbev/msag101,,contains DNBSEQ-T7 libraries that should maybe...
1269,SRP044761,Evolutionary changes in cis regulation are ass...,Expression profiling by high-throughput sequen...,SRA,total,Bgee 1K,27,,,GSE59707,PRJNA256000,,https://kuleuven.limo.libis.be/discovery/fulld...,,,"no paper, seems likely this is part of a thesi..."


### add annotations to git

In [25]:
! git pull

Already up to date.


In [26]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [27]:
! git add $git_experiment_path $git_library_path

In [28]:
! git commit -m $commit_message_exp

[develop 32c06cb] adding annotated bulk experiment SRP044761
 2 files changed, 28 insertions(+)


In [29]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 3.44 KiB | 3.44 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   ea17dc7..32c06cb  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push